In [1]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'   # 0 = all messages, 1 = filter INFO, 2 = filter WARNING, 3 = filter ERROR
import tensorflow as tf
# … rest of your imports and code …

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# --- 1) Định nghĩa các trọng số (theo đúng kích thước bạn đã cho) ---
# Wx: (hidden_size=3, input_size=4)
Wx = tf.constant([[ 1,  0, -1,  1],
                  [ 0,  2,  1,  0],
                  [ 1,  2,  1,  0]], dtype=tf.float32)
# Wh: (hidden_size=3, hidden_size=3)
Wh = tf.constant([[1, 0, 1],
                  [0, 1, 1],
                  [0, 1, 0]], dtype=tf.float32)
# bh: (hidden_size=3,)
bh = tf.constant([1, -1, -1], dtype=tf.float32)

# Wy: (output_size=1, hidden_size=3)
Wy = tf.constant([[ 3, 0, -1]], dtype=tf.float32)
# by: (output_size=1,)
by = tf.constant([3.], dtype=tf.float32)


# --- 2) Xây dựng model Keras ---
#   - SimpleRNN: units=3, activation='tanh', return_sequences=True để lấy H[t] cho mỗi bước
#   - Dense: 1 neuron, activation='sigmoid' cho đầu ra Y[t]

rnn = layers.SimpleRNN(
    units=3,
    activation='tanh',
    return_sequences=True,
    # kernel: shape (input_dim, units)  = (4,3)  ← Wx^T
    kernel_initializer=keras.initializers.Constant(tf.transpose(Wx)),
    # recurrent_kernel: shape (units, units) = (3,3) ← Wh^T
    recurrent_initializer=keras.initializers.Constant(tf.transpose(Wh)),
    # bias: (units,) = bh
    bias_initializer=keras.initializers.Constant(bh),
)

dense = layers.Dense(
    units=1,
    activation='sigmoid',
    # kernel: shape (units_in, units_out) = (3,1) ← Wy^T
    kernel_initializer=keras.initializers.Constant(tf.transpose(Wy)),
    bias_initializer=keras.initializers.Constant(by),
)

model = keras.Sequential([rnn, dense])


# --- 3) Chuẩn bị dữ liệu vào và chạy forward ---
X_seq = [
    [2, -1,  0, 1],
    [1,  0,  1, 0],
    [1,  1,  0, 0],
    [1,  2, -1, 2],
]
# thêm batch dimension: shape = (1, time_steps=4, input_dim=4)
X = tf.constant([X_seq], dtype=tf.float32)

# forward pass
Y = model(X)  # shape = (1, 4, 1)

# flatten và in ra
print("Y[t] = ", tf.reshape(Y, [-1]).numpy())


Y[t] =  [0.99884164 0.9960476  0.9944871  0.993336  ]
